# Alt V2 paired decision checks

July 24–September 3, 2026. Sources captured September 4, 21:58–22:01Z. Run from this notebook directory using Python 3.11+. No network, database, pipeline, notification or production writes. The linked `reproduce.py` contains the full matching, frozen-proof validation, scoring and close-selection code; this execution regenerates only this evidence packet’s derived files.

Prospectively frozen decisions are scored as hypothetical 1u exposures at recorded locked odds. Fees, slippage and allocated operating costs are unmeasured. Historical +38.585u is not part of this sample.

In [1]:
from pathlib import Path
import contextlib, io, json, runpy
root = Path.cwd()
assert (root / 'reproduce.py').is_file()
with contextlib.redirect_stdout(io.StringIO()):
    runpy.run_path(str(root / 'reproduce.py'), run_name='__main__')
a = json.loads((root / 'analysis.json').read_text())
print('Offline reproduction passed; selected proofs:', a['checks']['selected_proofs_valid'])
print('Lock/history/frozen mismatches:', a['checks']['lock_history_frozen_mismatches'])

Offline reproduction passed; selected proofs: 113
Lock/history/frozen mismatches: []


## Complete populations and paired exposure

726 graded locked rows from 727 operational locks. 720 frozen V2 states include one unselected row without a history outcome. Seven graded locks have no frozen V2 state. Unknown outcomes are excluded, not scored as zero. Mainline FIRE uses official size; all 91 are FIRE 1u in this window.

In [2]:
s = a['summary']
assert s['common_v2_universe_flat']['rows'] + s['no_frozen_v2']['rows'] == s['mainline_all_locked_flat']['rows'] == 726
assert s['all_alt']['rows'] + s['unselected_v2']['rows'] + s['pending_v2']['rows'] == 719
assert abs(s['all_alt']['pnl'] - s['consensus_core']['pnl'] - s['reentry_expansion']['pnl']) < 0.000002
assert s['alt_mainline_overlap']['rows'] + s['alt_incremental_lean']['rows'] == 113
assert s['alt_mainline_overlap']['rows'] + s['mainline_fire_not_alt']['rows'] == 91
for name in ['mainline_fire_weighted','mainline_all_locked_flat','all_alt','consensus_core','reentry_expansion','alt_mainline_overlap','alt_incremental_lean','mainline_fire_not_alt']:
    print(name, s[name])
print('Portfolio P&L difference, unequal exposure:', round(s['all_alt']['pnl'] - s['mainline_fire_weighted']['pnl'], 6))
print('Missing canonical outcome:', a['checks']['missing_history'])

mainline_fire_weighted {'losses': 44, 'pitchers': 66, 'pnl': -9.085141, 'roi_pct': -9.9837, 'rows': 91, 'settled_risk_units': 91, 'slate_dates': 40, 'voids': 0, 'wins': 47}
mainline_all_locked_flat {'losses': 358, 'pitchers': 189, 'pnl': -43.490061, 'roi_pct': -6.0152, 'rows': 726, 'settled_risk_units': 723, 'slate_dates': 42, 'voids': 3, 'wins': 365}
all_alt {'losses': 63, 'pitchers': 85, 'pnl': -24.889187, 'roi_pct': -22.2225, 'rows': 113, 'settled_risk_units': 112, 'slate_dates': 38, 'voids': 1, 'wins': 49}
consensus_core {'losses': 38, 'pitchers': 63, 'pnl': -11.089692, 'roi_pct': -14.5917, 'rows': 77, 'settled_risk_units': 76, 'slate_dates': 35, 'voids': 1, 'wins': 38}
reentry_expansion {'losses': 25, 'pitchers': 29, 'pnl': -13.799495, 'roi_pct': -38.3319, 'rows': 36, 'settled_risk_units': 36, 'slate_dates': 25, 'voids': 0, 'wins': 11}
alt_mainline_overlap {'losses': 12, 'pitchers': 24, 'pnl': -1.979032, 'roi_pct': -7.6117, 'rows': 26, 'settled_risk_units': 26, 'slate_dates': 20, 

## Diversity and rolling stability

Use the same 42-date slate calendar, including zero-selection dates. These 29 overlapping 14-slate windows are descriptive and not independent replications. Required slices include unknown attribution. Path B comes from later lock-consistent archive context, not a proven frozen covariate.

In [3]:
assert len(a['checks']['complete_window_slate_dates']) == 42
assert len(a['rolling_14_slate_windows']) == 29
for name in ['consensus_core','reentry_expansion']:
    for field, buckets in a['slices'][name].items():
        assert sum(v['rows'] for v in buckets.values()) == s[name]['rows'], (name, field)
    print(name, {k:v for k,v in a['robustness'][name].items() if k != 'by_slate'})
print('Latest common 14-slate window:', a['rolling_14_slate_windows'][-1])

consensus_core {'leave_one_slate_out_pnl_range': [-12.571905, -7.089692], 'lock_minutes_range': [24.250340433333335, 29.855369166666666], 'max_pitcher_rows': 3, 'positive_14_windows': 2, 'rolling_14_windows': 29, 'unique_pitchers': 63}
reentry_expansion {'leave_one_slate_out_pnl_range': [-15.745421, -10.799495], 'lock_minutes_range': [24.149117783333335, 29.787065466666665], 'max_pitcher_rows': 3, 'positive_14_windows': 0, 'rolling_14_windows': 29, 'unique_pitchers': 29}
Latest common 14-slate window: {'end': '2026-09-03', 'scores': {'all_alt': {'losses': 24, 'pitchers': 35, 'pnl': -13.736391, 'roi_pct': -37.1254, 'rows': 37, 'settled_risk_units': 37, 'slate_dates': 13, 'voids': 0, 'wins': 13}, 'consensus_core': {'losses': 13, 'pitchers': 22, 'pnl': -6.116391, 'roi_pct': -26.593, 'rows': 23, 'settled_risk_units': 23, 'slate_dates': 11, 'voids': 0, 'wins': 10}, 'mainline_all_locked_flat': {'losses': 114, 'pitchers': 150, 'pnl': -6.490834, 'roi_pct': -2.7504, 'rows': 236, 'settled_risk_u

## Bounded close provenance and costs

The existing exact-match producer uses a 20-minute pre-lock quote window and a 20-minute pregame close freshness window. Provider comes from the frozen proof. This is a selected-only paper process packet, not a new close target population or accepted-bet CLV. Ten targets have no retained snapshots; the remaining exclusions can have snapshots that fail exact provenance.

In [4]:
m = a['checks']['close_manifest']
assert m['eligible_close_rows'] + m['excluded_lock_rows'] == 113
assert m['eligible_close_rows'] == 90 and m['database_writes'] == 0
assert sum(m['exclusion_reason_counts'].values()) == 23
print({k:m[k] for k in ['eligible_close_rows','excluded_lock_rows','exclusion_reason_counts','max_close_age_minutes','max_lock_provenance_age_minutes']})
for lane in ['consensus_core','reentry_expansion']:
    print(lane, a['slices'][lane]['final_clv'])
print('Costs: quoted payout only; execution fees, slippage and allocated infrastructure costs unmeasured.')

{'eligible_close_rows': 90, 'excluded_lock_rows': 23, 'exclusion_reason_counts': {'missing_lock_provenance_snapshot': 22, 'missing_pregame_close_snapshot': 1}, 'max_close_age_minutes': 20, 'max_lock_provenance_age_minutes': 20}
consensus_core {'beat_close_price': {'losses': 10, 'pitchers': 17, 'pnl': -4.910976, 'roi_pct': -28.8881, 'rows': 17, 'settled_risk_units': 17, 'slate_dates': 13, 'voids': 0, 'wins': 7}, 'neutral_close': {'losses': 20, 'pitchers': 32, 'pnl': -8.313186, 'roi_pct': -23.0922, 'rows': 36, 'settled_risk_units': 36, 'slate_dates': 19, 'voids': 0, 'wins': 16}, 'unknown': {'losses': 6, 'pitchers': 19, 'pnl': 2.070862, 'roi_pct': 11.5048, 'rows': 19, 'settled_risk_units': 18, 'slate_dates': 13, 'voids': 1, 'wins': 12}, 'worse_close_price': {'losses': 2, 'pitchers': 5, 'pnl': 0.063609, 'roi_pct': 1.2722, 'rows': 5, 'settled_risk_units': 5, 'slate_dates': 4, 'voids': 0, 'wins': 3}}
reentry_expansion {'beat_close_price': {'losses': 1, 'pitchers': 1, 'pnl': -1.0, 'roi_pct': 

## Source-file integrity

Hashes identify captured subset files. The full hosted history payload has its own separate hash. Preserve the source files; future live reads belong in a new dated packet.

In [5]:
import hashlib
for name, expected in a['checks']['source_hashes'].items():
    assert hashlib.sha256((root / name).read_bytes()).hexdigest() == expected
    print(name, expected)
print('Hosted full-history artifact hash:', a['checks']['history_artifact_sha256'])
print('All notebook assertions passed.')

archive-context.json d404b6dfbaa981927515e2f0e44f22bdd552c85d054551e007d0debcbcb2131a
bounded-close-snapshots.json 86e997093c64a1db8a0971dded7bc6ac30c84d06318fdf4f320f33dbfb642c87
bounded-history.json 479ef2016fb1f8e0742b26f59c4c16aed8a0a92572fdcc46bee4663001ac39d6
comparison-universe.json cfb8f2c43678209cb1f6b3c5543d35f77dbb87405455b58f43e07bd5618bf32f
selected-lock-proofs.json f5fdaeffdd2b3bf31692a821073bdbe9ddcb2002081c8ca76652daa97c93d4f9
Hosted full-history artifact hash: 5c18c351f08886d40c6063c6d3c3eb37dc38ca3566facd2f1deeef800d4caeef
All notebook assertions passed.
